# PhishGuard-PK | Step 49: Fresh phishing URL source test

**Run all:** no GPU, no separate data ZIP, no unsafe link opening. The notebook downloads ONLY official OpenPhish/Tranco publisher feeds and tests URL strings offline. Results are research diagnostics, **not validated phishing detection accuracy**.

In [ ]:
from pathlib import Path
root=Path('/content/step49_phishguard')
(root/'scripts').mkdir(parents=True,exist_ok=True)
(root/'phishguard').mkdir(parents=True,exist_ok=True)
(root/'phishguard'/'__init__.py').write_text('',encoding='utf-8')
(root/'phishguard'/'url_policy.py').write_text('"""Step 44: conservative, offline URL-structure assessment.\n\nNever fetches links, resolves DNS, expands shorteners, or authenticates a domain.\nThe indicator is not a probability of phishing. Reference domains exist ONLY\nfor lookalike detection, never for automatically declaring a URL safe.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom urllib.parse import urlsplit\nimport ipaddress\nimport re\n\n# A small, dated reference set, NOT a complete official registry / allowlist.\n# All reference domain comparisons require a dot boundary.\nREFERENCE_DOMAINS = {\n    \'github\': (\'github.com\', \'github.io\'),\n    \'wikipedia\': (\'wikipedia.org\', \'wikimedia.org\'),\n    \'python\': (\'python.org\', \'pypi.org\'),\n    \'google\': (\'google.com\', \'google.com.pk\', \'googleapis.com\', \'googleusercontent.com\'),\n    \'microsoft\': (\'microsoft.com\', \'microsoftonline.com\'),\n    \'bisp\': (\'bisp.gov.pk\',),\n    \'jazzcash\': (\'jazzcash.com.pk\',),\n    \'meezan\': (\'meezanbank.com\', \'meezanbank.pk\'),\n    \'fbr\': (\'fbr.gov.pk\',),\n    \'hbl\': (\'hbl.com\',),\n    \'alfalah\': (\'bankalfalah.com\',),\n    \'easypaisa\': (\'easypaisa.com.pk\',),\n}\n# A namespace marker, not an assertion that every *.gov.pk site is authentic.\n# Used solely to detect gov.pk appended to an unrelated destination hostname.\nPUBLIC_SECTOR_NAMESPACE_MARKERS = (\'gov.pk\',)\nSHORTENERS = frozenset({\'bit.ly\', \'tinyurl.com\', \'t.co\', \'cutt.ly\', \'is.gd\', \'rb.gy\', \'ow.ly\', \'shorturl.at\'})\nAUTH_ACTION = re.compile(r\'(?i)(?:^|[/?#&=_-])(?:login|signin|sign-in|verify|verification|reset-password|password|otp|kyc|claim-prize|claim)(?:$|[/?#&=_-])\')\nSCAM_BAIT = re.compile(r\'(?i)(?:^|[/?#&=_-])(?:prize|winner|free-money|claim-reward|cash-reward|bisp-payment|8171-payment)(?:$|[/?#&=_-])\')\n\n\ndef _is_domain_or_subdomain(host: str, domain: str) -> bool:\n    return host == domain or host.endswith(\'.\' + domain)\n\n\ndef _brand_host_signal(host: str):\n    """(impersonation severity, reason) based on embedded brand/domain markers.\n\n    No ownership claim: official domains may be compromised and lookalike sites\n    may be benign. Signals are intentionally conservative.\n    """\n    for marker in PUBLIC_SECTOR_NAMESPACE_MARKERS:\n        if _is_domain_or_subdomain(host, marker):\n            continue\n        if host.startswith(marker + \'.\') or (\'.\' + marker + \'.\') in host or host.startswith(marker + \'-\'):\n            return 88, \'Hostname includes a government-style .gov.pk marker but points to another domain.\'\n        dashed = marker.replace(\'.\', \'-\')\n        if (host.startswith(dashed + \'.\') or host.startswith(dashed + \'-\')) and re.search(\n            r\'(?i)(?:verify|payment|8171|challan|account|login|claim)\', host\n        ):\n            return 82, \'Hostname imitates a government domain spelling with payment or verification language.\'\n    for brand, official_domains in REFERENCE_DOMAINS.items():\n        if any(_is_domain_or_subdomain(host, domain) for domain in official_domains):\n            continue\n        # An entire reference domain prepended to a different hostname is a\n        # strong deceptive-host signal: github.com.evil.org != github.com.\n        if any(host.startswith(domain + \'.\') or (\'.\' + domain + \'.\') in host\n               or host.startswith(domain + \'-\') for domain in official_domains):\n            return 88, f\'Domain resembles {brand} but the actual hostname is different; check the full hostname.\'\n        # Entire official domain with dots swapped for hyphens, for example\n        # bisp-gov-pk.verify.example. Strong spoof cue only when combined with\n        # account/payment context; do not classify all mentions as phishing.\n        for domain in official_domains:\n            dashed = domain.replace(\'.\', \'-\')\n            if (host.startswith(dashed + \'.\') or host.startswith(dashed + \'-\')) and re.search(\n                r\'(?i)(?:login|verify|payment|8171|claim|otp|kyc|prize|reward|account)\', host\n            ):\n                return 82, f\'Hostname uses a hyphenated spelling of a {brand} reference with a sensitive action.\'\n        # Bounded brand tokens with login / verification cues can be suspicious;\n        # ordinary brand-related fan/project domains are not automatically scams.\n        if re.search(r\'(?i)(?:^|[.-])\' + re.escape(brand) + r\'(?:[.-]|$)\', host):\n            if re.search(r\'(?i)(?:login|verify|secure|claim|payment|support|kyc|account|prize|8171|reward|bonus)\', host):\n                return 68, f\'Hostname combines a {brand} reference with account or verification language.\'\n            return 33, f\'Hostname mentions {brand}, but ownership is not verified.\'\n    return 0, \'\'\n\n\n@dataclass(frozen=True)\nclass URLAssessment:\n    indicator: int\n    reasons: tuple[str, ...]\n    hostname: str\n    status: str\n    high_confidence_structure: bool\n\n\ndef assess_url(url: str) -> URLAssessment:\n    """Returns interpretable offline indicator 0-100; NOT a fraud probability."""\n    if not isinstance(url, str) or not url.strip() or len(url) > 10000:\n        return URLAssessment(70, (\'URL could not be parsed safely.\',), \'\', \'invalid\', True)\n    raw = url.strip()\n    candidate = raw if re.match(r\'(?i)^https?://\', raw) else \'http://\' + raw\n    try:\n        parts = urlsplit(candidate)\n        host = (parts.hostname or \'\').strip(\'.\').lower()\n        port = parts.port  # validates invalid port values\n        if not host or any(c.isspace() for c in host) or \'\\\\\' in raw or \'%\' in host:\n            raise ValueError(\'Invalid hostname\')\n        # Convert unicode domains for reliable ASCII label processing.\n        ascii_host = host.encode(\'idna\').decode(\'ascii\')\n        if len(ascii_host) > 253 or any(len(label) > 63 or not label for label in ascii_host.split(\'.\')):\n            raise ValueError(\'Invalid DNS labels\')\n    except (ValueError, UnicodeError):\n        return URLAssessment(70, (\'Malformed or ambiguous URL structure. Avoid opening it.\',), \'\', \'invalid\', True)\n\n    score = 8\n    reasons = []\n    high = False\n    brand_score, brand_reason = _brand_host_signal(ascii_host)\n    if brand_score:\n        score = max(score, brand_score)\n        reasons.append(brand_reason)\n        high |= brand_score >= 75\n\n    if parts.username is not None or parts.password is not None or \'@\' in parts.netloc:\n        score = max(score, 82)\n        reasons.append(\'URL contains @ / userinfo: the apparent domain before @ may not be the destination.\')\n        high = True\n\n    ip = None\n    try:\n        ip = ipaddress.ip_address(ascii_host.strip(\'[]\'))\n    except ValueError:\n        pass\n    if ip:\n        if ip.is_global:\n            score = max(score, 45)\n            reasons.append(\'Destination uses a numeric IP rather than a domain name.\')\n        else:\n            score = max(score, 28)\n            reasons.append(\'Destination is an internal, private or reserved IP address; cannot verify it as a public site.\')\n\n    if ascii_host in SHORTENERS:\n        score = max(score, 48)\n        reasons.append(\'Shortened link hides its final destination; it has not been expanded.\')\n\n    if \'xn--\' in ascii_host or any(ord(c) > 127 for c in host):\n        score = max(score, 48)\n        reasons.append(\'Internationalized / punycode hostname; visually similar letters may be confusing.\')\n\n    if len(ascii_host.split(\'.\')) >= 5:\n        score = max(score, 31)\n        reasons.append(\'Hostname has several subdomain levels; inspect its real ending carefully.\')\n\n    # Only a compounded, contextual signal. HTTP alone is not phishing;\n    # HTTPS alone is not proof of legitimacy.\n    pathquery = (parts.path or \'\') + \'?\' + (parts.query or \'\')\n    action = bool(AUTH_ACTION.search(pathquery))\n    bait = bool(SCAM_BAIT.search(pathquery))\n    if action and parts.scheme.lower() == \'http\':\n        score = max(score, 52)\n        reasons.append(\'Account/verification action on unencrypted HTTP link.\')\n    elif action and score >= 33:\n        score = max(score, 53)\n        reasons.append(\'Account/verification action adds risk to an already unusual hostname.\')\n    if bait and score >= 33:\n        score = max(score, 58)\n        reasons.append(\'Prize/reward keywords appear with other URL warning signs.\')\n    if port not in (None, 80, 443):\n        score = max(score, 34)\n        reasons.append(\'URL uses a nonstandard network port.\')\n    if len(raw) > 250:\n        score = max(score, 33)\n        reasons.append(\'Unusually long link, inspect the complete destination.\')\n\n    if not reasons:\n        reasons.append(\'No strong offline structural warning found; this does NOT verify the website is safe.\')\n    status = \'high_structural_risk\' if score >= 75 else (\'caution\' if score >= 45 else \'low_indicators_not_verified\')\n    return URLAssessment(min(100, int(score)), tuple(reasons), ascii_host, status, high)\n',encoding='utf-8')
(root/'scripts'/'step49_external_url_evaluation.py').write_text('"""Step49: SOURCE-CLAIMED phishing and popular-domain proxy checks.\n\nCompares local URL structural warnings against two documented *different*\nsources. NEVER opens candidate pages, resolves domains, or fetches redirect\nchains. Network mode downloads ONLY two hard-coded publisher datasets, NOT\nany URL contained in them. No model/threshold selection against these data.\n\nThis is independent-source *diagnostics*, NOT a validated phishing classifier.\nThe Tranco set is a popularity proxy, not a collection of authenticated\nlegitimate webpages. Phishing URLs can occur on trusted hostnames.\n\nOffline usage:\n python scripts/step49_external_url_evaluation.py \\\\\n  --phishing-file path/to/OpenPhish_feed.txt \\\\\n  --proxy-file path/to/Tranco_list.csv --output /private/results\n\nDownload mode (safe publisher feeds only):\n python scripts/step49_external_url_evaluation.py --download --output /private/results\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport random\nimport sys\nimport urllib.request\nimport zipfile\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom urllib.parse import urlsplit\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom phishguard.url_policy import assess_url\n\n# Both values derived from public publishers; do not accept arbitrary fetch URLs.\nOPENPHISH_SOURCE = \'https://raw.githubusercontent.com/openphish/public_feed/refs/heads/main/feed.txt\'\nTRANCO_ID = \'V354N\'  # list created 2026-11-01 (see https://tranco-list.eu/list/V354N)\nTRANCO_SOURCE = f\'https://tranco-list.eu/download_daily/{TRANCO_ID}\'\nSOURCES = {\'openphish\': OPENPHISH_SOURCE, \'tranco\': TRANCO_SOURCE}\nACCEPTED_TYPES = (\'.txt\', \'.csv\', \'.zip\')\nMAX_BYTES = 38 * 1024 * 1024\nN_LIMIT = 400\n\n\ndef _sha(value: str) -> str:\n    return hashlib.sha256(value.encode(\'utf-8\')).hexdigest()\n\n\ndef _download_only_publisher(which: str) -> bytes:\n    """Fixed URL only. NEVER follows a URL *from* either publisher feed."""\n    url = SOURCES[which]\n    req = urllib.request.Request(url, headers={\n        \'User-Agent\': \'PhishGuard-PK-academic-offline-evaluation/1.0\',\n        \'Accept\': \'text/plain,text/csv,application/zip,application/octet-stream\',\n    })\n    # Publisher URL fixed; Python may follow publisher HTTP redirects.\n    # Never pass any malicious/test URL to urlopen.\n    class SameHostOnlyRedirect(urllib.request.HTTPRedirectHandler):\n        def redirect_request(self, req, fp, code, msg, headers, newurl):\n            # Keep requests on the selected publisher host; never follow an\n            # arbitrary malicious redirect when downloading threat feeds.\n            newparts=urlsplit(newurl)\n            if newparts.scheme!=\'https\' or newparts.hostname!=urlsplit(url).hostname:\n                raise ValueError(\'Publisher attempted a cross-host / non-HTTPS redirect\')\n            return super().redirect_request(req,fp,code,msg,headers,newurl)\n    opener=urllib.request.build_opener(SameHostOnlyRedirect())\n    with opener.open(req, timeout=35) as res:\n        data = res.read(MAX_BYTES + 1)\n    if len(data) > MAX_BYTES:\n        raise ValueError(\'Publisher feed exceeds size cap\')\n    return data\n\n\ndef _parse_urls_from_openphish(raw: bytes) -> list[str]:\n    contents = raw.decode(\'utf-8-sig\', errors=\'replace\')\n    seen = set()\n    out = []\n    for line in contents.splitlines():\n        u = line.strip()\n        if not u or u.startswith(\'#\') or u in seen or len(u) > 10000:\n            continue\n        if urlsplit(u).scheme.lower() not in (\'http\',\'https\'):\n            continue\n        if not urlsplit(u).hostname:\n            continue\n        seen.add(u)\n        out.append(u)\n    return out\n\n\ndef _parse_tranco(raw: bytes) -> list[tuple[int, str]]:\n    if raw[:4] == b\'PK\\x03\\x04\':\n        with zipfile.ZipFile(io.BytesIO(raw)) as z:\n            valid = [i for i in z.infolist() if i.filename.lower().endswith(\'.csv\') and not i.is_dir()]\n            if not valid or len(valid) != 1 or valid[0].file_size > MAX_BYTES * 4:\n                raise ValueError(\'Invalid Tranco ZIP structure\')\n            with z.open(valid[0]) as f:\n                content = f.read(MAX_BYTES * 4 + 1)\n                if len(content) > MAX_BYTES * 4:\n                    raise ValueError(\'Tranco uncompressed size cap\')\n    else:\n        content = raw\n    result = []\n    for row in csv.reader(io.StringIO(content.decode(\'utf-8-sig\', errors=\'replace\'))):\n        if len(row) < 2:\n            continue\n        try:\n            rank = int(row[0].strip())\n        except ValueError:\n            continue\n        domain = row[1].strip().lower().strip(\'.\')\n        if not domain or \'/\' in domain or \':\' in domain or \'@\' in domain or \' \' in domain:\n            continue\n        try:\n            domain.encode(\'idna\')\n        except UnicodeError:\n            continue\n        # Tranco entries are base domains; their homepage is only a *proxy*.\n        result.append((rank,domain))\n    return result\n\n\ndef _sample_phishing(urls: list[str], n: int) -> list[str]:\n    # Uniform sample by hostname, one example per hostname; avoid 200 URLs\n    # from the same compromised domain counting as 200 independent successes.\n    by_host: dict[str,str] = {}\n    for url in urls:\n        hostname = (urlsplit(url).hostname or \'\').lower()\n        if hostname not in by_host:\n            by_host[hostname] = url\n    hosts = sorted(by_host)\n    random.Random(20261102).shuffle(hosts)\n    return [by_host[h] for h in hosts[:n]]\n\n\ndef _sample_tranco(rows: list[tuple[int, str]], n: int) -> list[tuple[int,str]]:\n    # Draw a reproducible subset of the *top 100k* domains. This is not a\n    # verified non-phishing set and contains only base-domain homepages.\n    seen = set()\n    eligible=[]\n    for rank,domain in rows:\n        if rank > 100000 or domain in seen:\n            continue\n        seen.add(domain)\n        eligible.append((rank,domain))\n    eligible.sort()\n    random.Random(4902).shuffle(eligible)\n    return eligible[:n]\n\n\ndef _score_cases(urls: list[str], kind: str, rank_map: dict[str,int]|None=None):\n    out = []\n    for url in urls:\n        result=assess_url(url)\n        # Do not include real malicious URLs, full domains, paths, screenshots,\n        # query strings, passwords, or token values in any result artifacts.\n        host = (urlsplit(url).hostname or \'\').lower()\n        row={\'record_hash\':_sha(url), \'host_hash\':_sha(host), \'cohort\':kind,\n             \'score\':result.indicator, \'status\':result.status,\n             \'warning_45\':result.indicator>=45, \'warning_75\':result.indicator>=75,\n             \'rank\':(rank_map or {}).get(host,\'\'),\n             \'signals\':len(result.reasons)}\n        out.append(row)\n    return out\n\n\ndef _metrics(rows:list[dict], positive:bool):\n    n=len(rows)\n    return {\'n\':n, \'warning_45_n\':sum(x[\'warning_45\'] for x in rows),\n            \'warning_75_n\':sum(x[\'warning_75\'] for x in rows),\n            \'warning_45_rate\':round(sum(x[\'warning_45\'] for x in rows)/n,4) if n else None,\n            \'warning_75_rate\':round(sum(x[\'warning_75\'] for x in rows)/n,4) if n else None,\n            \'interpretation\':(\'phishing feed catch rate, NOT verified universal recall\' if positive\n                 else \'warning frequency on popular-domain homepages, NOT false-positive rate on authenticated safe sites\')}\n\n\ndef evaluate(openphish_raw:bytes|None, tranco_raw:bytes|None, output:Path,\n             source_mode:str, n:int=N_LIMIT, errors:dict[str,str]|None=None) -> dict:\n    output.mkdir(parents=True,exist_ok=True)\n    report={\'experiment\':\'STEP49_FRESH_SOURCE_URL_DIAGNOSTIC_NOT_FINAL_MODEL_EVALUATION\',\n            \'timestamp_utc\':datetime.now(timezone.utc).isoformat(timespec=\'seconds\'),\n            \'mode\':source_mode, \'tranco_list_id\':TRANCO_ID,\n            \'sources\':SOURCES, \'fixed_thresholds_not_tuned_on_test\':[45,75],\n            \'record_selection\':\'deterministic host-diverse OpenPhish sample, deterministic top-100k Tranco domain homepage sample\',\n            \'any_candidate_URL_opened\':False,\n            \'safety\':\'No candidate URL requests, DNS resolution or content fetch. Only publishers fetched when --download set.\',\n            \'source_and_label_limits\':[\n              \'OpenPhish is provider-labelled feed, not a separately human-authenticated Pakistani phishing dataset.\',\n              \'Tranco rank is not proof of domain safety, and homepage proxies underrepresent legitimate deep paths.\',\n              \'Matching host names between two sources need investigation; do not infer ground truth from either alone.\',\n              \'This is not a random representative sample of 2026 Pakistani SMS phishing or current web traffic.\',\n              \'Not a calibration study, no confidence intervals or F1/accuracy claims, no trained model deployment allowed.\'\n            ],\n            \'url_model_deployment_approved\':False,\n            \'independent_50_100_real_SMS_evaluation_done\':False,\n            \'publisher_errors\':errors or {}}\n    rows=[]\n    host_sets={}\n    for name, raw in [(\'openphish\',openphish_raw),(\'tranco\',tranco_raw)]:\n        if raw is None:\n            report[name]={\'status\':\'NOT_AVAILABLE\',\'n\':0}\n            continue\n        report[f\'{name}_publisher_bytes_sha256\']=hashlib.sha256(raw).hexdigest()\n        if name==\'openphish\':\n            candidates=_parse_urls_from_openphish(raw)\n            chosen=_sample_phishing(candidates,n)\n            curr=_score_cases(chosen,\'OpenPhish_feed_label\')\n            host_sets[name]={urlsplit(x).hostname.lower() for x in chosen}\n            report[name]={\'status\':\'SOURCE_LABELLED_NOT_INDEPENDENTLY_REVIEWED\',\'raw_valid_unique_urls\':len(candidates),\n                          \'selected_distinct_hosts\':len(chosen),**_metrics(curr,positive=True)}\n        else:\n            candidates=_parse_tranco(raw)\n            chosen=_sample_tranco(candidates,n)\n            rank_map={d:i for i,d in chosen}\n            curr=_score_cases([\'https://\'+domain+\'/\' for i,domain in chosen],\'Tranco_top100k_homepage_proxy\',rank_map)\n            host_sets[name]={d for _,d in chosen}\n            report[name]={\'status\':\'POPULAR_DOMAIN_PROXY_NOT_VERIFIED_BENIGN\',\'raw_parsed_domains\':len(candidates),\n                          \'selected_distinct_hosts\':len(chosen),**_metrics(curr,positive=False)}\n        rows+=curr\n    if len(host_sets)==2:\n        report[\'selected_hostname_overlap_n\']=len(host_sets[\'openphish\'] & host_sets[\'tranco\'])\n    if rows:\n        path=output/\'STEP49_HASHED_CASES_ONLY.csv\'\n        with path.open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as f:\n            w=csv.DictWriter(f,fieldnames=list(rows[0]))\n            w.writeheader();w.writerows(rows)\n    report[\'result_status\']=\'INCOMPLETE_SOURCE_FETCH\' if len(host_sets)<2 else \'RESEARCH_DIAGNOSTIC_COMPLETE_NO_DEPLOYMENT\'\n    (output/\'STEP49_SOURCE_DIAGNOSTIC.json\').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    return report\n\n\ndef main(args=None):\n    p=argparse.ArgumentParser()\n    g=p.add_mutually_exclusive_group(required=True)\n    g.add_argument(\'--download\', action=\'store_true\',help=\'download only two fixed publisher feeds\')\n    g.add_argument(\'--phishing-file\',type=Path,help=\'downloaded public OpenPhish feed locally\')\n    p.add_argument(\'--proxy-file\',type=Path,help=\'local Tranco CSV or ZIP; required with --phishing-file\')\n    p.add_argument(\'--output\',type=Path,default=Path.cwd()/\'step49_private_output\')\n    p.add_argument(\'--n\',type=int,default=N_LIMIT)\n    a=p.parse_args(args)\n    if not 1<=a.n<=2000:\n        p.error(\'--n must be 1..2000\')\n    if a.phishing_file and not a.proxy_file:\n        p.error(\'--proxy-file required with --phishing-file\')\n    raw={};errors={}\n    if a.download:\n        for which in SOURCES:\n            try:\n                raw[which]=_download_only_publisher(which)\n            except Exception as exc:\n                errors[which]=f\'{type(exc).__name__}: {str(exc)[:200]}\'\n                raw[which]=None\n    else:\n        raw={\'openphish\':a.phishing_file.read_bytes(),\'tranco\':a.proxy_file.read_bytes()}\n    report=evaluate(raw[\'openphish\'],raw[\'tranco\'],a.output,\n                    \'LIVE_PUBLISHER_FEEDS\' if a.download else \'SUPPLIED_PUBLISHER_FILES\',a.n,errors)\n    print(json.dumps({key:report[key] for key in [\'result_status\',\'openphish\',\'tranco\',\'publisher_errors\',\'url_model_deployment_approved\']},indent=2))\n    return report\n\nif __name__==\'__main__\':\n    main()\n',encoding='utf-8')
print('Offline URL policy and fixed-source evaluation code loaded. No model changes.')

In [ ]:
import subprocess,sys
p=subprocess.run([sys.executable,str(root/'scripts'/'step49_external_url_evaluation.py'),'--download','--n','400','--output',str(root/'private_results')],capture_output=True,text=True,timeout=150)
print(p.stdout)
if p.stderr: print('Runner details:',p.stderr[-2500:])
print('Process exit:',p.returncode)
assert (root/'private_results'/'STEP49_SOURCE_DIAGNOSTIC.json').exists(),'No report produced; see runner details above'

In [ ]:
import json,zipfile
from google.colab import files
result=json.loads((root/'private_results'/'STEP49_SOURCE_DIAGNOSTIC.json').read_text(encoding='utf-8'))
print('RESULT STATUS:',result['result_status'])
print('OpenPhish:',result.get('openphish'))
print('Tranco:',result.get('tranco'))
print('Publisher errors:',result.get('publisher_errors'))
print('WARNING: This is NOT final verified phishing accuracy. URL model deployment approved:',result['url_model_deployment_approved'])
name='/content/PhishGuard_Step49_External_Research_Results.zip'
with zipfile.ZipFile(name,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for item in root.joinpath('private_results').iterdir():
        if item.is_file(): z.write(item,arcname=item.name)
print('Download research results:',name)
files.download(name)